In [1]:
from constrerl.erl_schema import (
    entity_labels,
    relations,
    
)
from constrerl.annotator import Annotator, AnnotationTypes, article_to_sentences
from constrerl.annotation_model import (
    AnnotatedArticle,
    load_collection,
    Entity    
)

In [2]:
dev_articles=load_collection("Dev")
train_articles=load_collection("Train")

In [3]:
train_human ={id:train_article for id, train_article in train_articles.items() if train_article.metadata.annotator != "distant"}
print(f"Train human articles: {len(train_human)}")
print(f"Train distant articles: {len(train_articles)-len(train_human)}")

Train human articles: 1949
Train distant articles: 2972


In [4]:
import datasets

In [5]:
label_list = ["O", "B-NE", "I-NE"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [6]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"

tokenizer = AutoTokenizer.from_pretrained(model_name, )

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
def convert_to_token_labels(
    text: str, entities: list[Entity], tokenizer: AutoTokenizer, label2id: dict
):
    encoding = tokenizer(
        text, return_offsets_mapping=True, padding='max_length', max_length=128, truncation=True
    )

    offsets = encoding["offset_mapping"]
    labels = ["O"] * len(offsets)

    for ent in entities:
        for i, (token_start, token_end) in enumerate(offsets):
            if token_start >= ent.end_idx or token_end <= ent.start_idx:
                continue

            if token_start == ent.start_idx:
                labels[i] = "B-NE"
            else:
                labels[i] = "I-NE"
    # convert to ids + mask special tokens
    label_ids = []
    for i, label in enumerate(labels):
        if offsets[i] == (0, 0):  # special tokens like [CLS]
            label_ids.append(-100)
        else:
            label_ids.append(label2id[label])

    encoding["labels"] = label_ids
    return encoding

In [8]:
import tqdm


def gen_dataset(
    articles: dict[int, AnnotatedArticle], tokenizer, label2id
):
    for article in tqdm.tqdm(articles.values()):
        sentences = article_to_sentences(article.metadata, entities=article.entities)
        for sentence in sentences:
            
            text = sentence.text

            encoding = convert_to_token_labels(text, sentence.entities, tokenizer, label2id)
            yield encoding

In [9]:
import pandas as pd
train_df = pd.DataFrame(gen_dataset(train_articles, tokenizer, label2id))
eval_df = pd.DataFrame(gen_dataset(dev_articles, tokenizer, label2id))


100%|██████████| 80/80 [00:00<00:00, 717.44it/s]


In [10]:
train_df

,input_ids,token_type_ids,attention_mask,offset_mapping,labels
0,"[2, 7760, 10998, 1682, 2743, 5468, 3028, 8275,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[(0, 0), (0, 3), (4, 14), (15, 17), (18, 23), ...","[-100, 1, 2, 0, 1, 2, 2, 2, 0, 0, 0, 0, 0, 0, ..."
1,"[2, 5199, 1682, 1680, 7760, 6512, 4085, 4442, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[(0, 0), (0, 11), (12, 14), (15, 18), (19, 22)...","[-100, 0, 0, 0, 1, 2, 2, 2, 0, 0, 0, 0, 1, 2, ..."
2,"[2, 6512, 4085, 5345, 1748, 2170, 1955, 2743, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[(0, 0), (0, 9), (10, 19), (20, 28), (29, 33),...","[-100, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 2, 0, 1, ..."
3,"[2, 2564, 663, 5661, 2281, 3186, 2520, 21, 228...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[(0, 0), (0, 8), (9, 10), (10, 12), (13, 18), ...","[-100, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."
4,"[2, 5169, 1955, 5864, 3321, 1690, 1680, 10998,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[(0, 0), (0, 12), (13, 20), (21, 28), (29, 44)...","[-100, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...
53240,"[2, 1680, 7692, 1685, 31, 50, 33, 5835, 14578,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[(0, 0), (0, 3), (4, 13), (14, 16), (17, 18), ...","[-100, 0, 0, 0, 0, 0, 0, 1, 2, 2, 2, 0, 0, 0, ..."
53241,"[2, 1680, 7692, 1685, 31, 50, 33, 4811, 20396,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[(0, 0), (0, 3), (4, 13), (14, 16), (17, 18), ...","[-100, 0, 0, 0, 0, 0, 0, 1, 2, 2, 0, 0, 0, 0, ..."
53242,"[2, 31, 50, 33, 10277, 7423, 31, 18, 50, 33, 7...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[(0, 0), (0, 1), (1, 2), (2, 3), (3, 7), (7, 1...","[-100, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, ..."
53243,"[2, 1680, 2727, 14641, 1012, 3503, 1682, 1680,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[(0, 0), (0, 3), (4, 9), (10, 13), (13, 14), (...","[-100, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, ..."


In [11]:
train_dataset = datasets.Dataset.from_pandas(train_df)
eval_dataset = datasets.Dataset.from_pandas(eval_df)

In [12]:
train_dataset

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'labels'],
    num_rows: 53245
})

In [13]:
len(train_dataset[3]["attention_mask"])

128

In [14]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./finetuned/ned",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=5,    
    warmup_ratio=0.1,                   # critical for stability with higher LR
    weight_decay=0.01,
    # evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=200,
    eval_steps=500,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

trainer.train()

/tmp/ipykernel_3920732/1886032811.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
200,0.686800
400,0.255900
600,0.200100
800,0.175800
1000,0.170000
1200,0.157800
1400,0.159900
1600,0.158200
1800,0.145700
2000,0.153900


TrainOutput(global_step=16640, training_loss=0.10356241981857098, metrics={'train_runtime': 629.2169, 'train_samples_per_second': 423.105, 'train_steps_per_second': 26.446, 'total_flos': 1.73910894958656e+16, 'train_loss': 0.10356241981857098, 'epoch': 5.0})

In [15]:
# annotate dev articles with the trained model

import torch
from constrerl.sentences import annotated_sentences_to_article


def detect_entities(
    article: AnnotatedArticle,
    model: AutoModelForTokenClassification,
    tokenizer: AutoTokenizer,
):
    sentences = article_to_sentences(
        article.metadata,
    )
    entities: list[Entity] = []
    for sentence in sentences:
        text = sentence.text
        tokens = tokenizer(
            text,
            return_offsets_mapping=True,
            padding="max_length",
            max_length=128,
            truncation=True,
        )
        input_ids = (
            torch.tensor(tokens["input_ids"]).unsqueeze(0).to(model.device)
        )  # batch size 1
        attention_mask = (
            torch.tensor(tokens["attention_mask"]).unsqueeze(0).to(model.device)
        )
        predictions = model(input_ids=input_ids, attention_mask=attention_mask)
        predicted_labels = predictions.logits.argmax(dim=-1).squeeze().tolist()
        print(predicted_labels)
        offsets = tokens["offset_mapping"]
        entities_sentence: list[Entity] = []
        current_entity: Entity = None
        last_end_idx = 0
        is_running = False
        def finish_current_entity():
            nonlocal current_entity, last_end_idx, is_running
            if current_entity is not None and is_running:
                current_entity.end_idx = last_end_idx
                current_entity.text_span = text[
                    current_entity.start_idx : current_entity.end_idx
                ]
                entities_sentence.append(current_entity)
                current_entity = None
        for label_id, (start, end) in zip(predicted_labels, offsets):
            if label_id == label2id["O"]:
                finish_current_entity()
                is_running = False
                continue
            if label_id == label2id["B-NE"]:
                finish_current_entity()
                current_entity = Entity(
                    start_idx=start,
                    end_idx=end,
                    label="NE",
                    location="title" if sentence.title else "abstract",
                    text_span=text[start:end],
                )
                is_running = True
                last_end_idx = end
            if label_id == label2id["I-NE"] or label_id == label2id["B-NE"]:
                last_end_idx = end
        finish_current_entity()
        filtered_ents = []
        for ent in entities_sentence:
            if ent.text_span.strip() != "":
                ent.end_idx = ent.end_idx - 1
                filtered_ents.append(ent)
            sentence.entities = filtered_ents
    return annotated_sentences_to_article(sentences, article.metadata)


ned_dev_articles = {}
annotated_articles = {}
for id, article in dev_articles.items():
    article_cp = AnnotatedArticle(**article.__dict__)
    for ent in article_cp.entities:
        ent.label = "NE"
    annotated_article = detect_entities(article_cp, model, tokenizer)
    ned_dev_articles[id] = article_cp
    annotated_articles[id] = annotated_article
    print(f"Article {article.metadata.title} detected entities:")
    for ent in annotated_article.entities:
        print(f" - {ent.text_span} ({ent.start_idx}-{ent.end_idx})")

[0, 1, 2, 0, 1, 2, 2, 2, 0, 1, 2, 2, 0, 1, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 2, 2, 2, 0, 0, 0, 2, 0, 0, 1, 2, 0, 2, 0, 1, 1, 2, 0, 0, 2, 2, 2, 2, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 2, 2, 2, 2, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 2, 2, 2, 0, 1, 2, 2, 2, 0, 1, 2, 2, 2, 0, 1, 0, 0, 0, 2, 0, 0, 1]
[0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 2, 2, 2, 2, 0, 1, 0, 0, 0, 1, 2, 2, 0, 1, 0, 0, 1, 2, 2, 2, 2, 2, 0, 1, 2, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 1, 2, 2, 2, 2, 0, 0, 0, 0, 0, 1, 1, 2, 2, 0, 1, 0, 0, 1, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0]
[0, 1, 2, 0, 0, 1, 2, 0, 0, 0, 1, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 0, 0, 0, 1, 2, 0, 0, 1, 2, 0, 1, 1, 1, 1, 0, 0, 0, 0, 2, 2, 2, 0, 0, 0, 0

In [16]:
from constrerl.utils import prepare_for_eval
from constrerl.eval.evaluate import eval_submission_NED

## Current best:
Full Train + Martinelli params for NED
|   |precision	|recall	    |f1	        |micro_precision	    |micro_recall	|micro_f1	|type
|--|           --|      --|         --|                     --|             --|             --|--
|0	|0.773364	|0.810789	|0.791634	|0.773364	            |0.810789	    |0.791634	|NE

In [17]:


def tuple_to_scores(tup):
    # precision, recall, f1, micro_precision, micro_recall, micro_f1
    return {
        "precision": tup[0],
        "recall": tup[1],
        "f1": tup[2],
        "micro_precision": tup[3],
        "micro_recall": tup[4],
        "micro_f1": tup[5],
    }


# ground_truth_test = {"test": test_article}
eval_functions = {
    "NE": eval_submission_NED,
}
scores_list = []
for typ, eval_fun in eval_functions.items():
    print(f"Evaluating {typ}...")
    scores = eval_fun(
        prepare_for_eval(annotated_articles), prepare_for_eval(ned_dev_articles)
    )
    scores=tuple_to_scores(scores)
    scores["type"] = typ
    scores_list.append(scores)

import pandas as pd
df_scores = pd.DataFrame(scores_list)
df_scores

Evaluating NE...


,precision,recall,f1,micro_precision,micro_recall,micro_f1,type
0,0.774573,0.809599,0.791699,0.774573,0.809599,0.791699,NE


In [18]:
ned_dev_articles['34912029'].entities

[Entity(start_idx=4, end_idx=42, location='abstract', text_span='human Apolipoprotein E4 (ApoE4) variant', label='NE', uri='http://purl.obolibrary.org/obo/NCIT_C105362'),
 Entity(start_idx=91, end_idx=109, location='abstract', text_span="Alzheimer's disease", label='NE', uri='http://purl.obolibrary.org/obo/NCIT_C2866'),
 Entity(start_idx=112, end_idx=113, location='abstract', text_span='AD', label='NE', uri='http://purl.obolibrary.org/obo/NCIT_C2866'),
 Entity(start_idx=117, end_idx=123, location='abstract', text_span='Cadmium', label='NE', uri='http://purl.obolibrary.org/obo/CHEBI_22977'),
 Entity(start_idx=126, end_idx=127, location='abstract', text_span='Cd', label='NE', uri='http://purl.obolibrary.org/obo/CHEBI_22977'),
 Entity(start_idx=198, end_idx=237, location='abstract', text_span='humanized ApoE4 knock-in (ApoE4-KI) mice', label='NE', uri='http://purl.obolibrary.org/obo/NCIT_C45247'),
 Entity(start_idx=254, end_idx=282, location='abstract', text_span='ApoE3 (common allele)-KI

In [19]:
annotated_articles['34912029'].entities

[Entity(start_idx=0, end_idx=15, location='title', text_span='Cadmium exposure', label='NE', uri=None),
 Entity(start_idx=4, end_idx=26, location='abstract', text_span='human Apolipoprotein E4', label='NE', uri=None),
 Entity(start_idx=29, end_idx=32, location='abstract', text_span='ApoE', label='NE', uri=None),
 Entity(start_idx=91, end_idx=109, location='abstract', text_span="Alzheimer's disease", label='NE', uri=None),
 Entity(start_idx=112, end_idx=113, location='abstract', text_span='AD', label='NE', uri=None),
 Entity(start_idx=117, end_idx=123, location='abstract', text_span='Cadmium', label='NE', uri=None),
 Entity(start_idx=126, end_idx=127, location='abstract', text_span='Cd', label='NE', uri=None),
 Entity(start_idx=198, end_idx=237, location='abstract', text_span='humanized ApoE4 knock-in (ApoE4-KI) mice', label='NE', uri=None),
 Entity(start_idx=254, end_idx=282, location='abstract', text_span='ApoE3 (common allele)-KI mice', label='NE', uri=None),
 Entity(start_idx=309, e

In [20]:
model.save_pretrained("./finetuned/ned/best_model_pubmed")

In [21]:
tokenizer.save_pretrained("./finetuned/ned/best_model_pubmed")

('./finetuned/ned/best_model_pubmed/tokenizer_config.json',
 './finetuned/ned/best_model_pubmed/special_tokens_map.json',
 './finetuned/ned/best_model_pubmed/vocab.txt',
 './finetuned/ned/best_model_pubmed/added_tokens.json',
 './finetuned/ned/best_model_pubmed/tokenizer.json')